# Game Intelligence AI — Stage 4: Dataset Preparation & Validation
**Purpose:** define the dataset contract, validate JSONL records, check provenance/license metadata, detect duplicates, create leakage-resistant grouped splits, and export a readiness report.

**Important:** this notebook does **not** train the model. Its built-in examples are original synthetic pipeline tests, not a meaningful training corpus. Do not mix them into a production training dataset unless you intentionally choose to do so.

**Revision 1:** the upload prompt has been removed from the automatic Run all path. This prevents Colab from waiting indefinitely at an interactive file picker.

## Safety and data policy
- Never scrape code or documentation without checking the applicable license and terms.
- A repository-level license may not cover every file, asset, level, or third-party dependency.
- Keep source URL, repository/revision, file path, license evidence, and a human review status for each imported record.
- This notebook uses a conservative license allowlist for a first-pass screen. Passing the screen is **not legal advice** and does not replace checking the original license and file-level exceptions.
- Do not upload model weights, adapters, private keys, or large datasets to the project GitHub repository.

In [ ]:
from pathlib import Path
import json, hashlib, re, random, csv, datetime, collections, os

BASE_DIR = Path("/content/game_intelligence_stage4")
INPUT_DIR = BASE_DIR / "input"
OUTPUT_DIR = BASE_DIR / "output"
INPUT_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Workspace:", BASE_DIR)
print("Input:", INPUT_DIR)
print("Output:", OUTPUT_DIR)

## 1. Dataset schema

Each JSONL line must be a JSON object with these required fields:

- `id`, `task_type`, `instruction`, `input_context`, `output`
- `language`, `engine_or_stack`, `difficulty`
- `license`, `source`, `split`

Additional provenance fields are strongly recommended:
`source_url`, `source_revision`, `source_file`, `license_evidence`,
`group_id`, `synthetic`, and `review_status`.

Allowed task categories are deliberately broad and can be extended after review.

In [ ]:
TASK_TYPES = {
    "game_programming", "game_design", "game_physics", "level_design",
    "npc_ai", "debugging", "optimization", "testing",
    "architecture", "tools_pipeline"
}
DIFFICULTIES = {"beginner", "intermediate", "advanced", "expert"}
SPLITS = {"train", "validation", "test"}
REQUIRED_FIELDS = {
    "id", "task_type", "instruction", "input_context", "output",
    "language", "engine_or_stack", "difficulty", "license", "source", "split"
}

# Conservative first-pass allowlist. Every source/file still needs human license review.
ALLOWED_LICENSES = {
    "MIT", "Apache-2.0", "BSD-2-Clause", "BSD-3-Clause",
    "CC0-1.0", "CC-BY-4.0"
}

def canonical_text(value):
    return re.sub(r"\s+", " ", str(value or "").strip()).lower()

def content_fingerprint(record):
    parts = [
        record.get("task_type", ""),
        record.get("instruction", ""),
        record.get("input_context", ""),
        record.get("output", "")
    ]
    return hashlib.sha256("\n".join(canonical_text(x) for x in parts).encode("utf-8")).hexdigest()

def validate_record(record):
    errors = []
    missing = REQUIRED_FIELDS - set(record)
    if missing:
        errors.append("missing fields: " + ", ".join(sorted(missing)))
        return errors

    if record["task_type"] not in TASK_TYPES:
        errors.append(f"unknown task_type: {record['task_type']}")
    if record["difficulty"] not in DIFFICULTIES:
        errors.append(f"unknown difficulty: {record['difficulty']}")
    if record["split"] not in SPLITS:
        errors.append(f"unknown split: {record['split']}")

    for field in ("id", "instruction", "output", "language", "engine_or_stack", "license", "source"):
        if not isinstance(record[field], str) or not record[field].strip():
            errors.append(f"{field} must be a non-empty string")

    if not isinstance(record["input_context"], str):
        errors.append("input_context must be a string")

    if record.get("synthetic") is True:
        if record.get("source") != "original_synthetic":
            errors.append("synthetic records must use source='original_synthetic'")
    else:
        for field in ("source_url", "source_revision", "source_file", "license_evidence", "group_id"):
            if not record.get(field):
                errors.append(f"non-synthetic record missing provenance field: {field}")
        if record.get("review_status") != "approved":
            errors.append("non-synthetic record has not passed human review")

    if record["license"] not in ALLOWED_LICENSES and record.get("synthetic") is not True:
        errors.append(f"license not in first-pass allowlist: {record['license']}")

    return errors

## 2. Create a tiny synthetic dataset to test the pipeline

These examples are deliberately short and original. They are useful for testing parsing, schema checks, duplicate detection, and split generation only. They are not enough to train a capable general game-development model.

In [ ]:
synthetic_records = [
    {
        "id": "syn-001", "task_type": "game_programming",
        "instruction": "Write a minimal game loop with a fixed update step.",
        "input_context": "Stack: plain JavaScript and HTML5 Canvas. No external engine.",
        "output": "Use requestAnimationFrame for rendering and accumulate elapsed time; run update with a fixed timestep while the accumulator exceeds the timestep.",
        "language": "English", "engine_or_stack": "JavaScript/HTML5 Canvas",
        "difficulty": "beginner", "license": "CC0-1.0",
        "source": "original_synthetic", "split": "train",
        "group_id": "synthetic-loop", "synthetic": True, "review_status": "approved"
    },
    {
        "id": "syn-002", "task_type": "game_physics",
        "instruction": "Explain how to prevent a platform character from falling through a floor.",
        "input_context": "A 2D character uses a rectangular collider and vertical velocity.",
        "output": "Resolve vertical motion against the floor: detect overlap after movement, place the character on the floor surface, set vertical velocity to zero, and mark the character grounded.",
        "language": "English", "engine_or_stack": "Engine-agnostic 2D",
        "difficulty": "beginner", "license": "CC0-1.0",
        "source": "original_synthetic", "split": "train",
        "group_id": "synthetic-collision", "synthetic": True, "review_status": "approved"
    },
    {
        "id": "syn-003", "task_type": "npc_ai",
        "instruction": "Describe a simple finite-state machine for a guard NPC.",
        "input_context": "States: patrol, investigate, chase, return.",
        "output": "Patrol along waypoints; switch to investigate when a sound is heard; switch to chase when the player is visible; return to patrol after losing the target and reaching the last patrol route.",
        "language": "English", "engine_or_stack": "Engine-agnostic",
        "difficulty": "intermediate", "license": "CC0-1.0",
        "source": "original_synthetic", "split": "validation",
        "group_id": "synthetic-guard-ai", "synthetic": True, "review_status": "approved"
    },
    {
        "id": "syn-004", "task_type": "debugging",
        "instruction": "Suggest a debugging plan for a player that moves faster on high-refresh-rate displays.",
        "input_context": "Movement is updated once per rendered frame using a constant position increment.",
        "output": "Measure frame delta, reproduce at multiple refresh rates, move using velocity multiplied by delta time or a fixed simulation timestep, and add a regression test comparing displacement over equal simulated time.",
        "language": "English", "engine_or_stack": "JavaScript",
        "difficulty": "intermediate", "license": "CC0-1.0",
        "source": "original_synthetic", "split": "test",
        "group_id": "synthetic-frame-rate", "synthetic": True, "review_status": "approved"
    },
    {
        "id": "syn-005", "task_type": "level_design",
        "instruction": "Give a basic checklist for reviewing a beginner platformer level.",
        "input_context": "The level introduces jumping, moving platforms, and a checkpoint.",
        "output": "Check that the safe route is readable, the jump is physically reachable, hazards are introduced before being combined, the checkpoint is placed before a difficult section, and failure recovery is quick.",
        "language": "English", "engine_or_stack": "Engine-agnostic",
        "difficulty": "beginner", "license": "CC0-1.0",
        "source": "original_synthetic", "split": "validation",
        "group_id": "synthetic-level-checklist", "synthetic": True, "review_status": "approved"
    }
]

synthetic_path = INPUT_DIR / "synthetic_pipeline_test.jsonl"
with synthetic_path.open("w", encoding="utf-8") as f:
    for row in synthetic_records:
        f.write(json.dumps(row, ensure_ascii=False) + "\n")

print("Wrote", len(synthetic_records), "synthetic records to", synthetic_path)

## 3. Load JSONL data

The synthetic test file created above is already saved in the input folder. The next cell only lists available JSONL files and will not open a file picker or wait for user input.

To add a real dataset later, place its `.jsonl` file in:
`/content/game_intelligence_stage4/input/`

Then rerun the file-discovery cell and the loading cell. Keep synthetic and real licensed datasets in separate files.

In [ ]:
# Non-blocking file discovery.
# This cell intentionally does NOT call files.upload(), so Run all will not pause for a file picker.
jsonl_files = sorted(INPUT_DIR.glob("*.jsonl"))
print(f"Found {len(jsonl_files)} JSONL file(s) in: {INPUT_DIR}")
for path in jsonl_files:
    print(f" - {path.name} ({path.stat().st_size:,} bytes)")

if not jsonl_files:
    print("No JSONL files found. The synthetic test file should have been created in the previous section.")
print("\nTo add a real dataset later, upload/place its .jsonl file into this input folder, then rerun this cell and the following load/validation cells.")

In [ ]:
def read_jsonl(path):
    rows, parse_errors = [], []
    with Path(path).open("r", encoding="utf-8") as f:
        for line_no, line in enumerate(f, start=1):
            if not line.strip():
                continue
            try:
                rows.append(json.loads(line))
            except Exception as exc:
                parse_errors.append({"file": str(path), "line": line_no, "error": str(exc)})
    return rows, parse_errors

all_rows, parse_errors = [], []
for path in sorted(INPUT_DIR.glob("*.jsonl")):
    rows, errs = read_jsonl(path)
    all_rows.extend(rows)
    parse_errors.extend(errs)

print("Records loaded:", len(all_rows))
print("JSON parse errors:", len(parse_errors))

## 4. Validate schema, provenance, and license metadata

In [ ]:
validation_errors = []
seen_ids = set()

for index, row in enumerate(all_rows):
    row_errors = validate_record(row)
    if row.get("id") in seen_ids:
        row_errors.append("duplicate id")
    seen_ids.add(row.get("id"))
    for err in row_errors:
        validation_errors.append({"record_index": index, "id": row.get("id"), "error": err})

print("Schema/provenance errors:", len(validation_errors))
for item in validation_errors[:30]:
    print(item)

## 5. Detect exact and near-duplicate candidates

This step detects exact normalized duplicates. It is not semantic deduplication; similar prompts with different wording still need a later similarity review.

In [ ]:
fingerprints = collections.defaultdict(list)
for row in all_rows:
    fingerprints[content_fingerprint(row)].append(row.get("id"))

duplicate_groups = [ids for ids in fingerprints.values() if len(ids) > 1]
print("Exact duplicate groups:", len(duplicate_groups))
for group in duplicate_groups[:20]:
    print(group)

## 6. Group-aware split audit

For production data, records sharing a project/source/conversation/template should share one `group_id`. A group must never appear in more than one split. The built-in synthetic examples are intentionally assigned different groups.

In [ ]:
group_splits = collections.defaultdict(set)
for row in all_rows:
    group = row.get("group_id") or row.get("source") or row.get("id")
    group_splits[group].add(row.get("split"))

split_leaks = {group: sorted(splits) for group, splits in group_splits.items() if len(splits) > 1}
print("Groups crossing multiple splits:", len(split_leaks))
for group, splits in split_leaks.items():
    print(group, "->", splits)

## 7. Dataset summary and readiness decision

A dataset is **not ready** if it contains parse errors, schema/provenance errors, duplicate IDs, exact duplicate content, or group leakage. This is only a technical gate; a human must still verify the actual license and content quality.

In [ ]:
counts_by_task = collections.Counter(row.get("task_type", "UNKNOWN") for row in all_rows)
counts_by_split = collections.Counter(row.get("split", "UNKNOWN") for row in all_rows)
synthetic_count = sum(row.get("synthetic") is True for row in all_rows)
real_count = len(all_rows) - synthetic_count

technical_ready = (
    len(all_rows) > 0
    and not parse_errors
    and not validation_errors
    and not duplicate_groups
    and not split_leaks
)

report = {
    "created_utc": datetime.datetime.now(datetime.timezone.utc).isoformat(),
    "records_total": len(all_rows),
    "synthetic_records": synthetic_count,
    "non_synthetic_records": real_count,
    "parse_errors": parse_errors,
    "validation_errors": validation_errors,
    "exact_duplicate_groups": duplicate_groups,
    "split_leaks": split_leaks,
    "counts_by_task": dict(counts_by_task),
    "counts_by_split": dict(counts_by_split),
    "technical_pipeline_ready": technical_ready,
    "training_authorized": False,
    "notes": [
        "This notebook does not train the model.",
        "Synthetic examples are pipeline tests, not a sufficient training corpus.",
        "Technical validation does not replace human license and content review.",
        "Do not begin a training run until the real dataset has been reviewed and approved."
    ]
}

report_path = OUTPUT_DIR / "stage4_dataset_validation_report.json"
report_path.write_text(json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8")

print(json.dumps({
    "technical_pipeline_ready": technical_ready,
    "records_total": len(all_rows),
    "synthetic_records": synthetic_count,
    "non_synthetic_records": real_count,
    "counts_by_task": dict(counts_by_task),
    "counts_by_split": dict(counts_by_split),
    "report_path": str(report_path)
}, ensure_ascii=False, indent=2))

## 8. Export the report

Download the report for your records. It is small and can be stored in the GitHub repository. Do not upload large datasets, adapters, or model weights to GitHub.

In [ ]:
try:
    from google.colab import files
    files.download(str(report_path))
except Exception:
    print("Report saved at:", report_path)

## Stage 4 exit criteria

Before moving to a training notebook, all of the following must be true:

1. Real records have traceable source URLs, revisions, file paths, and license evidence.
2. Every source/file has passed human license review.
3. No malformed records, duplicate IDs, or exact duplicate examples remain.
4. Related records are grouped so train/validation/test do not leak across groups.
5. Validation and test sets are held out and not used for training.
6. The dataset has meaningful coverage across the task categories, not just a handful of synthetic examples.
7. A human has sampled examples and confirmed that outputs are correct, useful, and safe.

**Current expected outcome with the built-in data:** the technical pipeline should pass, but the report must still say that the synthetic-only data is not an approved training dataset.